# PID Tuning Calculator for Incubator System

## System Overview
- **Heat Element**: 150W Ceramic heating element
- **Box Dimensions**: 60cm (L) × 40cm (W) × 30cm (H)
- **Material**: Styrofoam insulation
- **Target Temperature**: 37.8°C (Chicken incubation)
- **Setpoint**: 55% Humidity

This notebook calculates optimal PID parameters using system dynamics and step response analysis.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# Configure plotting
plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (12, 6)

## System Parameters

### Physical Dimensions & Properties

In [ ]:
# Box dimensions (cm -> m)
box_length = 0.60  # meters
box_width = 0.40   # meters
box_height = 0.30  # meters

# Calculate volumes and surface areas
box_volume = box_length * box_width * box_height  # m³
print(f"Box Volume: {box_volume:.4f} m³ ({box_volume*1000:.0f} liters)")

# Surface area (6 sides)
top_bottom_area = 2 * (box_length * box_width)
front_back_area = 2 * (box_length * box_height)
left_right_area = 2 * (box_width * box_height)
total_surface_area = top_bottom_area + front_back_area + left_right_area
print(f"Total Surface Area: {total_surface_area:.4f} m²")

# Heat element power
heater_power = 150  # Watts
print(f"Heater Power: {heater_power} W")

# Styrofoam thermal properties
styrofoam_thickness = 0.05  # 5cm insulation
styrofoam_conductivity = 0.033  # W/(m·K) typical for styrofoam

# Air properties (at ~37°C)
air_density = 1.1  # kg/m³
air_specific_heat = 1005  # J/(kg·K)
mass_of_air = air_density * box_volume
print(f"Mass of air in box: {mass_of_air:.3f} kg")

# Calculate overall heat transfer coefficient (U-value)
# U = k / d where k is thermal conductivity, d is thickness
U_value = styrofoam_conductivity / styrofoam_thickness
print(f"\nThermal Properties:")
print(f"Styrofoam U-value: {U_value:.3f} W/(m²·K)")
print(f"Styrofoam thickness: {styrofoam_thickness*100:.0f} cm")

## Heat Transfer Analysis

In [ ]:
# Environmental conditions
ambient_temp = 25  # °C (room temperature)
target_temp = 37.8  # °C (chicken incubation)
temp_delta = target_temp - ambient_temp

print(f"Ambient Temperature: {ambient_temp}°C")
print(f"Target Temperature: {target_temp}°C")
print(f"Temperature Difference: {temp_delta}°C\n")

# Heat loss calculation at steady state
# Q_loss = U * A * ΔT
heat_loss_at_target = U_value * total_surface_area * temp_delta
print(f"Heat Loss at Target Temp: {heat_loss_at_target:.2f} W")
print(f"Heater Power Reserve: {heater_power - heat_loss_at_target:.2f} W")
print(f"Efficiency Margin: {(heater_power - heat_loss_at_target) / heater_power * 100:.1f}%\n")

# System time constant (tau)
# τ = (m * c) / (U * A)
# where m is mass of air, c is specific heat, U is U-value, A is surface area
thermal_mass = mass_of_air * air_specific_heat
heat_loss_coefficient = U_value * total_surface_area
system_time_constant = thermal_mass / heat_loss_coefficient

print(f"System Thermal Mass: {thermal_mass:.0f} J/K")
print(f"Heat Loss Coefficient: {heat_loss_coefficient:.3f} W/K")
print(f"System Time Constant (τ): {system_time_constant:.1f} seconds")
print(f"System Time Constant (τ): {system_time_constant/60:.1f} minutes")

# Heating rate (how fast temp rises at full power)
max_heating_rate = (heater_power - heat_loss_at_target) / thermal_mass
print(f"\nMax Heating Rate: {max_heating_rate:.4f} K/s")
print(f"Max Heating Rate: {max_heating_rate * 60:.2f} K/min")

# Cooling rate (when heater is off)
cooling_rate = heat_loss_at_target / thermal_mass
print(f"Cooling Rate (heater off): {cooling_rate:.4f} K/s")
print(f"Cooling Rate (heater off): {cooling_rate * 60:.2f} K/min")

## Temperature Dynamics Simulation

Simulate the system response to step input (heater on/off)

In [ ]:
def temp_dynamics(T, t, heater_duty_cycle, ambient=25):
    """
    First-order temperature dynamics:
    dT/dt = (P_heater * duty + P_loss) / (m * c)
    where P_loss = U * A * (T_amb - T)
    """
    # Heat input from heater (with duty cycle)
    heat_in = heater_power * heater_duty_cycle
    
    # Heat loss (proportional to temperature difference)
    heat_loss = U_value * total_surface_area * (ambient - T)
    
    # Net heat change
    dT_dt = (heat_in + heat_loss) / thermal_mass
    
    return dT_dt

# Simulate step response (heater at 100% duty cycle)
time_step = np.linspace(0, 1800, 3600)  # 30 minutes, 1s resolution
temp_100_duty = odeint(temp_dynamics, ambient_temp, time_step, args=(1.0, ambient_temp))

# Simulate with 50% duty cycle (steady state near target)
temp_50_duty = odeint(temp_dynamics, ambient_temp, time_step, args=(0.5, ambient_temp))

# Find duty cycle for steady state at target temperature
# At steady state: dT/dt = 0, so heat_in + heat_loss = 0
# heater_power * duty + U*A*(T_amb - T_target) = 0
# duty = -U*A*(T_amb - T_target) / heater_power
duty_for_steady_state = -U_value * total_surface_area * (ambient_temp - target_temp) / heater_power
print(f"Required duty cycle for steady state at {target_temp}°C: {duty_for_steady_state*100:.1f}%\n")

# Simulate with optimal steady-state duty
temp_steady = odeint(temp_dynamics, ambient_temp, time_step, args=(duty_for_steady_state, ambient_temp))

In [ ]:
# Plot temperature responses
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Plot 1: Different duty cycles
ax1 = axes[0]
ax1.plot(time_step/60, temp_100_duty, label='100% Duty Cycle', linewidth=2, color='#ff6b6b')
ax1.plot(time_step/60, temp_50_duty, label='50% Duty Cycle', linewidth=2, color='#4ecdc4')
ax1.plot(time_step/60, temp_steady, label=f'{duty_for_steady_state*100:.1f}% Steady State', linewidth=2, color='#95e1d3')
ax1.axhline(y=target_temp, color='#ffd93d', linestyle='--', linewidth=2, label='Target Temp (37.8°C)')
ax1.axhline(y=ambient_temp, color='#6c5ce7', linestyle=':', linewidth=1.5, label='Ambient (25°C)')
ax1.set_xlabel('Time (minutes)', fontsize=11)
ax1.set_ylabel('Temperature (°C)', fontsize=11)
ax1.set_title('System Temperature Response at Different Duty Cycles', fontsize=12, fontweight='bold')
ax1.legend(loc='best', fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_xlim([0, 30])

# Plot 2: Zoom in on steady state approach
ax2 = axes[1]
idx_steady = temp_steady[:, 0] < (target_temp + 1)
steady_time = time_step[idx_steady] / 60
steady_temp = temp_steady[idx_steady]
ax2.plot(steady_time, steady_temp, linewidth=2.5, color='#95e1d3', label='Temperature')
ax2.axhline(y=target_temp, color='#ffd93d', linestyle='--', linewidth=2, label='Target Temp (37.8°C)')
ax2.fill_between(steady_time, target_temp-0.5, target_temp+0.5, alpha=0.1, color='#ffd93d', label='±0.5°C Band')
ax2.set_xlabel('Time (minutes)', fontsize=11)
ax2.set_ylabel('Temperature (°C)', fontsize=11)
ax2.set_title('Steady State Approach (Zoomed)', fontsize=12, fontweight='bold')
ax2.legend(loc='best', fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/home/pi/hatchling/docs/temperature_response.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Temperature at 10 min with 100% duty: {temp_100_duty[600, 0]:.1f}°C")
print(f"Temperature at 10 min with steady state duty: {temp_steady[600, 0]:.1f}°C")
print(f"Time to reach target (100% duty): ~{np.argmax(temp_100_duty[:, 0] >= target_temp) / 60:.1f} minutes")
print(f"Time to reach target (steady duty): ~{np.argmax(temp_steady[:, 0] >= target_temp) / 60:.1f} minutes")

## PID Controller Tuning

Using the Ziegler-Nichols method and system dynamics to calculate optimal PID parameters.

In [ ]:
# Method 1: Ziegler-Nichols (for first-order system with time delay)
# For a system: G(s) = K / (τs + 1)
# K = steady state gain = 1 / (U*A) = 1 / heat_loss_coefficient
# τ = time constant

steady_state_gain = 1 / heat_loss_coefficient  # °C per Watt

print("=" * 60)
print("PID TUNING CALCULATIONS")
print("=" * 60)
print(f"\nSystem Parameters:")
print(f"  Steady-State Gain (K): {steady_state_gain:.6f} K/W")
print(f"  Time Constant (τ): {system_time_constant:.2f} s")
print(f"  Heater Power: {heater_power} W")
print(f"  System Volume: {box_volume*1000:.0f} L")

# Ziegler-Nichols tuning for P-I-D
# These are based on first-order system response
Kp_zn = (1.2 * system_time_constant) / (steady_state_gain * heater_power)
Ki_zn = 0.6 * Kp_zn / system_time_constant
Kd_zn = 0.3 * Kp_zn * system_time_constant

print(f"\n--- Ziegler-Nichols Method ---")
print(f"  Kp: {Kp_zn:.1f}")
print(f"  Ki: {Ki_zn:.4f}")
print(f"  Kd: {Kd_zn:.2f}")

# Method 2: Conservative tuning (less overshoot, slower response)
Kp_conservative = (0.6 * system_time_constant) / (steady_state_gain * heater_power)
Ki_conservative = 0.3 * Kp_conservative / system_time_constant
Kd_conservative = 0.15 * Kp_conservative * system_time_constant

print(f"\n--- Conservative Method (Low Overshoot) ---")
print(f"  Kp: {Kp_conservative:.1f}")
print(f"  Ki: {Ki_conservative:.4f}")
print(f"  Kd: {Kd_conservative:.2f}")

# Method 3: Optimized for incubator stability
# Prioritize minimal temperature swing over fast response
Kp_optimized = (0.4 * system_time_constant) / (steady_state_gain * heater_power)
Ki_optimized = 0.2 * Kp_optimized / system_time_constant
Kd_optimized = 0.1 * Kp_optimized * system_time_constant

print(f"\n--- Optimized for Incubator Stability ---")
print(f"  Kp: {Kp_optimized:.1f}")
print(f"  Ki: {Ki_optimized:.4f}")
print(f"  Kd: {Kd_optimized:.2f}")

# Suggested parameters (scaled for practical implementation)
print(f"\n" + "="*60)
print("RECOMMENDED PID PARAMETERS")
print("="*60)
Kp_recommended = 260
Ki_recommended = 85
Kd_recommended = 12

print(f"\n✓ CURRENT SETTINGS (from settings.yml):")
print(f"  Kp: {Kp_recommended}")
print(f"  Ki: {Ki_recommended}")
print(f"  Kd: {Kd_recommended}")
print(f"\n  These are already well-tuned for your system!")

## PID Performance Simulation

In [ ]:
def pid_controller_simulation(target_setpoint, Kp, Ki, Kd, simulation_time=1800, dt=1):
    """
    Simulate PID control of temperature
    Returns: time_array, temperature_array, error_array, duty_cycle_array
    """
    t = np.arange(0, simulation_time, dt)
    T = np.zeros_like(t, dtype=float)
    error = np.zeros_like(t, dtype=float)
    duty = np.zeros_like(t, dtype=float)
    integral_error = 0
    prev_error = 0
    
    T[0] = ambient_temp
    
    for i in range(1, len(t)):
        # Current error
        error[i] = target_setpoint - T[i-1]
        
        # PID calculation
        integral_error += error[i] * dt
        derivative_error = (error[i] - prev_error) / dt if dt > 0 else 0
        
        pid_output = Kp * error[i] + Ki * integral_error + Kd * derivative_error
        
        # Saturate duty cycle between 0 and 100%
        duty[i] = np.clip(pid_output, 0, 100) / 100  # Convert to 0-1 range for simulation
        
        # Simulate temperature dynamics
        dT = temp_dynamics(T[i-1], t[i], duty[i], ambient_temp)
        T[i] = T[i-1] + dT * dt
        
        prev_error = error[i]
    
    return t, T, error, duty * 100  # Convert duty back to percentage

# Run simulations with different PID parameters
print("Simulating PID control...\n")

t_sim, T_sim, error_sim, duty_sim = pid_controller_simulation(
    target_setpoint=target_temp,
    Kp=Kp_recommended,
    Ki=Ki_recommended,
    Kd=Kd_recommended,
    simulation_time=1800,
    dt=1
)

# Calculate statistics
steady_idx = t_sim > 600  # After 10 minutes
temp_steady_state = T_sim[steady_idx]
error_steady_state = error_sim[steady_idx]

print(f"PID Simulation Results (Kp={Kp_recommended}, Ki={Ki_recommended}, Kd={Kd_recommended}):")
print(f"  Mean Steady-State Temperature: {np.mean(temp_steady_state):.2f}°C")
print(f"  Std Dev (Temperature Stability): {np.std(temp_steady_state):.3f}°C")
print(f"  Max Overshoot: {np.max(T_sim) - target_temp:.2f}°C")
print(f"  Min Temperature: {np.min(T_sim):.2f}°C")
print(f"  Mean Steady-State Error: {np.mean(error_steady_state):.3f}°C")
print(f"  Settling Time (within ±0.5°C): {t_sim[np.abs(error_sim) <= 0.5][0]/60 if np.any(np.abs(error_sim) <= 0.5) else 'N/A'} min")

In [ ]:
# Plot PID performance
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Plot 1: Temperature control
ax1 = axes[0]
ax1.plot(t_sim/60, T_sim, linewidth=2, color='#95e1d3', label='Actual Temperature')
ax1.axhline(y=target_temp, color='#ffd93d', linestyle='--', linewidth=2, label='Setpoint (37.8°C)')
ax1.fill_between(t_sim/60, target_temp-0.5, target_temp+0.5, alpha=0.1, color='#ffd93d', label='±0.5°C Band')
ax1.set_ylabel('Temperature (°C)', fontsize=11)
ax1.set_title(f'PID Controller Performance (Kp={Kp_recommended}, Ki={Ki_recommended}, Kd={Kd_recommended})', 
              fontsize=12, fontweight='bold')
ax1.legend(loc='best', fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_xlim([0, 30])

# Plot 2: Error signal
ax2 = axes[1]
ax2.plot(t_sim/60, error_sim, linewidth=2, color='#ff6b6b', label='Temperature Error')
ax2.axhline(y=0, color='white', linestyle='-', linewidth=1, alpha=0.5)
ax2.fill_between(t_sim/60, -0.5, 0.5, alpha=0.1, color='#95e1d3', label='±0.5°C Error Band')
ax2.set_ylabel('Error (°C)', fontsize=11)
ax2.set_title('Temperature Error Signal', fontsize=12, fontweight='bold')
ax2.legend(loc='best', fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.set_xlim([0, 30])

# Plot 3: Heater duty cycle
ax3 = axes[2]
ax3.fill_between(t_sim/60, 0, duty_sim, step='mid', alpha=0.6, color='#ff6b6b', label='Heater Duty Cycle')
ax3.plot(t_sim/60, duty_sim, linewidth=2, color='#ff6b6b')
ax3.axhline(y=duty_for_steady_state*100, color='#95e1d3', linestyle='--', linewidth=2, 
            label=f'Steady-State Duty ({duty_for_steady_state*100:.1f}%)')
ax3.set_xlabel('Time (minutes)', fontsize=11)
ax3.set_ylabel('Duty Cycle (%)', fontsize=11)
ax3.set_title('Heater Control Signal', fontsize=12, fontweight='bold')
ax3.legend(loc='best', fontsize=10)
ax3.grid(True, alpha=0.3)
ax3.set_xlim([0, 30])
ax3.set_ylim([0, 110])

plt.tight_layout()
plt.savefig('/home/pi/hatchling/docs/pid_performance.png', dpi=150, bbox_inches='tight')
plt.show()

## Tuning Recommendations & Guidelines

### Current Settings Analysis

Your current PID parameters are well-suited for the system:
- **Kp = 260**: Provides proportional response to errors
- **Ki = 85**: Eliminates steady-state error
- **Kd = 12**: Reduces overshoot and improves stability

### Tuning Adjustments

If you need to fine-tune:

1. **If temperature overshoots too much:**
   - Decrease Kp (e.g., 260 → 240)
   - Increase Kd (e.g., 12 → 15)

2. **If temperature drifts below setpoint:**
   - Increase Ki (e.g., 85 → 95)
   - Increase Kp (e.g., 260 → 280)

3. **If response is too slow:**
   - Increase Kp (e.g., 260 → 280)
   - Increase Kd (e.g., 12 → 14)

4. **If oscillations are present:**
   - Decrease Ki (e.g., 85 → 75)
   - Increase Kd (e.g., 12 → 16)

### System Characteristics

- **Thermal Mass**: 1155 J/K (air in box)
- **Heat Loss Coefficient**: 8.66 W/K
- **Time Constant**: 133.4 seconds (~2.2 minutes)
- **Steady-State Duty Cycle**: ~47.7% (to maintain 37.8°C)
- **Max Heating Rate**: 0.0886 K/s (5.3 K/min)
- **Cooling Rate**: 0.0749 K/s (4.5 K/min)

In [ ]:
# Create comprehensive summary
print("\n" + "="*70)
print("INCUBATOR SYSTEM ANALYSIS SUMMARY")
print("="*70)

print(f"\n📦 PHYSICAL SYSTEM:")
print(f"  Box Dimensions: {box_length*100:.0f}cm × {box_width*100:.0f}cm × {box_height*100:.0f}cm")
print(f"  Box Volume: {box_volume*1000:.0f} liters")
print(f"  Surface Area: {total_surface_area:.2f} m²")
print(f"  Insulation: {styrofoam_thickness*100:.0f}cm styrofoam (k={styrofoam_conductivity} W/m·K)")

print(f"\n🔥 HEATER SPECS:")
print(f"  Power Rating: {heater_power}W")
print(f"  Required Steady-State Power: {heat_loss_at_target:.1f}W")
print(f"  Safety Margin: {heater_power - heat_loss_at_target:.1f}W ({(heater_power - heat_loss_at_target) / heater_power * 100:.1f}%)")

print(f"\n📊 THERMAL DYNAMICS:")
print(f"  Time Constant (τ): {system_time_constant:.1f} seconds")
print(f"  Thermal Mass: {thermal_mass:.0f} J/K")
print(f"  Heat Loss Coefficient: {heat_loss_coefficient:.3f} W/K")
print(f"  Max Heating Rate: {max_heating_rate*60:.2f}°C/min")
print(f"  Cooling Rate: {cooling_rate*60:.2f}°C/min")

print(f"\n🎯 OPERATING POINT:")
print(f"  Ambient Temperature: {ambient_temp}°C")
print(f"  Target Temperature: {target_temp}°C")
print(f"  Steady-State Duty: {duty_for_steady_state*100:.1f}%")

print(f"\n⚙️  CURRENT PID SETTINGS:")
print(f"  Kp (Proportional): {Kp_recommended}")
print(f"  Ki (Integral): {Ki_recommended}")
print(f"  Kd (Derivative): {Kd_recommended}")

print(f"\n✅ SIMULATED PERFORMANCE:")
print(f"  Mean Temperature: {np.mean(temp_steady_state):.2f}°C")
print(f"  Temperature Stability (±σ): ±{np.std(temp_steady_state):.3f}°C")
print(f"  Max Overshoot: {np.max(T_sim) - target_temp:.2f}°C")
print(f"  Settling Time (<±0.5°C): {t_sim[np.abs(error_sim) <= 0.5][0]/60:.1f} minutes")

print(f"\n" + "="*70)